In [ ]:
from flask import Flask, request, jsonify
import threading
import math
from datetime import datetime

# ==============================
# FILTERS
# ==============================

# Moving Average (Flex + FSR)
def moving_average(values, window=5):
    if not values:
        return []
    result = []
    for i in range(len(values)):
        start = max(0, i - window + 1)
        avg = sum(values[start:i+1]) / (i - start + 1)
        result.append(avg)
    return result

# Low-pass filter (EMG + IMU)
def low_pass(values, alpha=0.1):
    if not values:
        return []
    filtered = [values[0]]
    for i in range(1, len(values)):
        filtered.append(alpha * values[i] + (1 - alpha) * filtered[-1])
    return filtered

# ==============================
# APPLY FILTERS TO DATA
# ==============================
def apply_filters(data):
    if not data:
        return []

    # extract columns safely
    flex1 = [d.get("flex1",0) for d in data]
    flex2 = [d.get("flex2",0) for d in data]
    fsr1  = [d.get("fsr1",0) for d in data]
    fsr2  = [d.get("fsr2",0) for d in data]
    emg   = [d.get("emg",0) for d in data]
    ax = [d.get("ax",0) for d in data]
    ay = [d.get("ay",0) for d in data]
    az = [d.get("az",0) for d in data]
    gx = [d.get("gx",0) for d in data]
    gy = [d.get("gy",0) for d in data]
    gz = [d.get("gz",0) for d in data]

    # Apply filters
    flex1_f = moving_average(flex1)
    flex2_f = moving_average(flex2)
    fsr1_f  = moving_average(fsr1)
    fsr2_f  = moving_average(fsr2)
    emg_f   = low_pass(emg)
    ax_f = low_pass(ax)
    ay_f = low_pass(ay)
    az_f = low_pass(az)
    gx_f = low_pass(gx)
    gy_f = low_pass(gy)
    gz_f = low_pass(gz)

    filtered_data = []
    for i in range(len(data)):
        filtered_data.append({
            "time": data[i].get("time",0),
            "flex1": flex1_f[i],
            "flex2": flex2_f[i],
            "fsr1": fsr1_f[i],
            "fsr2": fsr2_f[i],
            "emg": emg_f[i],
            "ax": ax_f[i],
            "ay": ay_f[i],
            "az": az_f[i],
            "gx": gx_f[i],
            "gy": gy_f[i],
            "gz": gz_f[i],
        })
    return filtered_data

# ==============================
# FEATURE FUNCTIONS
# ==============================
def accel_mag(d):
    return math.sqrt(d["ax"]**2 + d["ay"]**2 + d["az"]**2)

def compute_speed(data):
    if not data:
        return 0
    # Use average acceleration magnitude to reduce drift
    mags = [accel_mag(d) for d in data]
    return sum(mags)/len(mags) if mags else 0

def compute_rom(data):
    if not data:
        return 0
    angle = 0
    prev = data[0]
    angles = []
    for i in range(1, len(data)):
        dt = max((data[i]["time"] - prev["time"])/1000,0)
        angle += data[i]["gx"] * dt
        angle = max(min(angle,90),-90)  # clamp to reduce spikes/drift
        angles.append(angle)
        prev = data[i]
    return max(angles) - min(angles) if angles else 0

def compute_trajectory(data):
    if not data or len(data)<2:
        return 0
    vectors = []
    for i in range(1, len(data)):
        dx = data[i]["ax"] - data[i-1]["ax"]
        dy = data[i]["ay"] - data[i-1]["ay"]
        dz = data[i]["az"] - data[i-1]["az"]
        mag = math.sqrt(dx**2 + dy**2 + dz**2)
        if mag < 1e-3:  # ignore tiny movements
            continue
        vectors.append((dx/mag, dy/mag, dz/mag))
    if len(vectors)<2:
        return 0
    dots = [sum(vectors[i][j]*vectors[i-1][j] for j in range(3)) for i in range(1,len(vectors))]
    return sum(dots)/len(dots) if dots else 0

def compute_deviation(data):
    if not data:
        return 0
    return max([accel_mag(d) for d in data]) - min([accel_mag(d) for d in data])

def compute_flex(data):
    if not data:
        return 0
    f1 = [d["flex1"] for d in data]
    f2 = [d["flex2"] for d in data]
    return (sum(f1)+sum(f2))/(len(f1)+len(f2))

def compute_force(data):
    if not data:
        return 0
    f1 = [d["fsr1"] for d in data]
    f2 = [d["fsr2"] for d in data]
    return (sum(f1)+sum(f2))/(len(f1)+len(f2))

def compute_emg(data):
    if not data:
        return 0
    # Use RMS instead of simple average to reduce spike effect
    vals = [d["emg"] for d in data]
    square_sum = sum([v**2 for v in vals])
    return math.sqrt(square_sum/len(vals)) if vals else 0

# ==============================
# ASSESSMENT
# ==============================
def assess_reach(speed, rom, traj, dev):
    return rom > 60 and 1 < speed < 3 and traj > 0.85 and dev < 2000

def assess_grasp(force, flex):
    return force > 50 and flex > 40

def assess_manipulation(traj, duration):
    return traj > 0.8 and duration < 6

def assess_release(force, flex):
    return force < 20 and flex < 20

def assess_session(data):
    data = apply_filters(data)

    speed = compute_speed(data)
    rom = compute_rom(data)
    traj = compute_trajectory(data)
    dev = compute_deviation(data)
    flex = compute_flex(data)
    force = compute_force(data)
    emg = compute_emg(data)
    duration = (data[-1]["time"] - data[0]["time"])/1000

    print("\n===== FEATURES =====")
    print(f"Speed: {speed:.2f}")
    print(f"ROM: {rom:.2f}")
    print(f"Trajectory: {traj:.2f}")
    print(f"Deviation: {dev:.2f}")
    print(f"Flex: {flex:.2f}")
    print(f"Force: {force:.2f}")
    print(f"EMG: {emg:.2f}")

    # Assessment
    results = {
        "Reach": assess_reach(speed, rom, traj, dev),
        "Grasp": assess_grasp(force, flex),
        "Manipulation": assess_manipulation(traj, duration),
        "Release": assess_release(force, flex)
    }

    print("\n===== RESULTS =====")
    for k,v in results.items():
        print(f"{k}: {'PASS' if v else 'FAIL'}")

    needed_functions = [k for k,v in results.items() if not v]

    if not needed_functions:
        print("\nFINAL: No Exercises Needed ")
    else:
        print("\nFINAL: Needs Training ")
        print("\nPatient Needs to Train:")
        for f in needed_functions:
            print(f"- {f}")

# ==============================
# FLASK SERVER
# ==============================
app = Flask(__name__)

@app.route("/data", methods=["POST"])
def receive_data():
    payload = request.json
    if not payload:
        return jsonify({"status":"error","message":"No data received"}),400

    assess_session(payload)
    return jsonify({"status":"received"}),200

def run_flask():
    app.run(host="0.0.0.0", port=5000)

if __name__=="__main__":
    threading.Thread(target=run_flask).start()
    print("Flask server running, waiting for ESP data via WiFi...")